In [1]:
# --- Imports
import re
import json
import math
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, Any, List, Tuple

import warnings
warnings.filterwarnings("ignore")

In [2]:
# --- Config
RAW_TXT_PATH = "../data/raw/vehicle_knowledge_base.txt"         
OUT_DIR = Path("../data/processed")   # output folder

OUT_DIR.mkdir(parents=True, exist_ok=True)
# (OUT_DIR / "chunks").mkdir(parents=True, exist_ok=True)

In [3]:
# IO utils 
def read_text(path: str) -> str:
    return Path(path).read_text(encoding="utf-8").replace("\r\n", "\n").replace("\r", "\n")

def write_text(path: Path, text: str) -> None:
    path.write_text(text, encoding="utf-8")

def read_jsonl(path: str | Path) -> List[Dict[str, Any]]:
    """
        Read a JSONL file and return list of dicts.
    """
    path = Path(path)
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows

def write_jsonl(path: Path, rows: List[Dict[str, Any]]) -> None:
    with path.open("w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

In [4]:
# Title extraction: "The 2011 BMW 1 Series M is a ..."
TITLE_RE = re.compile(r"^(The\s+\d{4}\s+.+?)\s+is\s+a\s+", re.IGNORECASE)

def _make_doc_id(title: str, idx: int) -> str:
    safe = re.sub(r"[^a-zA-Z0-9]+", "_", title.strip()).strip("_").lower()
    return f"{safe}_{idx:05d}"

In [5]:
def parse_vehicle_docs(raw_txt_path: str) -> List[Dict[str, Any]]:
    raw = read_text(raw_txt_path)

    # Split on blank-line blocks.
    # This pattern splits when there are 2+ blank lines.
    blocks = [b.strip() for b in re.split(r"\n\s*\n\s*\n+", raw) if b.strip()]

    docs: List[Dict[str, Any]] = []
    for i, block in enumerate(blocks):
        first_line = block.splitlines()[0].strip()
        m = TITLE_RE.match(first_line)
        title = m.group(1).strip() if m else first_line[:80]

        doc_id = _make_doc_id(title, i)

        # Normalize whitespace inside each chunk (keeps chunk boundaries, improves retrieval)
        text = re.sub(r"[ \t]+", " ", block).strip()

        docs.append({"doc_id": doc_id, "title": title, "text": text})

    return docs

docs = parse_vehicle_docs(RAW_TXT_PATH)
print("Parsed vehicle docs:", len(docs))
print(docs[0] if docs else "No docs found")

Parsed vehicle docs: 11914
{'doc_id': 'the_2011_bmw_1_series_m_00000', 'title': 'The 2011 BMW 1 Series M', 'text': 'The 2011 BMW 1 Series M is a compact coupe.\nIt features a 6-cylinder premium unleaded engine producing 335 horsepower.\nThe transmission is manual with rear drive.\nIt has 2 doors.\nThis model falls under the following categories: factory tuner, luxury, high-performance.\nFuel efficiency is rated at 19 city MPG and 26 highway MPG.\nIt has a popularity score of 3916.\nThe MSRP for this vehicle is $46,135.'}


In [6]:
# Persist chunks 

#  Canonical JSONL (for later pipelines)
jsonl_path = OUT_DIR / "chunks.jsonl"
write_jsonl(jsonl_path, docs)
print("Wrote:", jsonl_path)

# Re-load canonical docs from disk so everything below uses persisted data
docs = read_jsonl(jsonl_path)
print("Reloaded docs from JSONL:", len(docs))

# -- each chunk in a different txt file
# one .txt per car (easy inspection / debugging)
# for d in docs:
#     write_text(OUT_DIR / "chunks" / f"{d['doc_id']}.txt", d["text"] + "\n")

# print("Wrote per-doc txt files to:", (OUT_DIR / "chunks").resolve())

Wrote: ..\data\processed\chunks.jsonl
Reloaded docs from JSONL: 11914


## Sparse Retrieval: BM25 


In [7]:
def simple_tokenize(text: str) -> List[str]:
    # Lowercase, keep alphanumerics as tokens
    return re.findall(r"[a-z0-9]+", text.lower())

class BM25Index:
    def __init__(self, docs: List[Dict[str, Any]], k1: float = 1.5, b: float = 0.75):
        self.docs = docs
        self.k1 = k1
        self.b = b

        self.doc_tokens = [simple_tokenize(d["text"]) for d in docs]
        self.doc_lens = [len(toks) for toks in self.doc_tokens]
        self.avgdl = (sum(self.doc_lens) / max(1, len(self.doc_lens)))

        # Document frequencies
        df = {}
        for toks in self.doc_tokens:
            for term in set(toks):
                df[term] = df.get(term, 0) + 1
        self.df = df
        self.N = len(docs)

        # IDF with BM25+ style smoothing
        self.idf = {}
        for term, n_q in df.items():
            self.idf[term] = math.log(1 + (self.N - n_q + 0.5) / (n_q + 0.5))

        # Term frequencies per doc
        self.tfs = []
        for toks in self.doc_tokens:
            tf = {}
            for t in toks:
                tf[t] = tf.get(t, 0) + 1
            self.tfs.append(tf)

    def score(self, query: str, doc_idx: int) -> float:
        q_tokens = simple_tokenize(query)
        tf = self.tfs[doc_idx]
        dl = self.doc_lens[doc_idx]
        score = 0.0

        for term in q_tokens:
            if term not in tf:
                continue
            idf = self.idf.get(term, 0.0)
            f = tf[term]
            denom = f + self.k1 * (1 - self.b + self.b * (dl / (self.avgdl or 1.0)))
            score += idf * (f * (self.k1 + 1)) / (denom or 1.0)

        return score

    def search(self, query: str, k: int = 5) -> List[Tuple[Dict[str, Any], float]]:
        scored = [(self.docs[i], self.score(query, i)) for i in range(self.N)]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]

# read from persisted JSONL to ensure we're using the same data as the FAISS index and later pipelines
docs = read_jsonl(OUT_DIR / "chunks.jsonl")

# Usage
bm25 = BM25Index(docs)

# Example query
bm25.search("rear drive manual 335 horsepower", k=3)

[({'doc_id': 'the_2014_bmw_z4_11896',
   'title': 'The 2014 BMW Z4',
   'text': 'The 2014 BMW Z4 is a compact convertible.\nIt features a 6-cylinder premium unleaded engine producing 335 horsepower.\nThe transmission is automated_manual with rear drive.\nIt has 2 doors.\nThis model falls under the following categories: luxury, high-performance.\nFuel efficiency is rated at 17 city MPG and 24 highway MPG.\nIt has a popularity score of 3916.\nThe MSRP for this vehicle is $65,800.'},
  7.340057241199167),
 ({'doc_id': 'the_2015_bmw_z4_11899',
   'title': 'The 2015 BMW Z4',
   'text': 'The 2015 BMW Z4 is a compact convertible.\nIt features a 6-cylinder premium unleaded engine producing 335 horsepower.\nThe transmission is automated_manual with rear drive.\nIt has 2 doors.\nThis model falls under the following categories: luxury, high-performance.\nFuel efficiency is rated at 17 city MPG and 24 highway MPG.\nIt has a popularity score of 3916.\nThe MSRP for this vehicle is $65,800.'},
  7.34

## Dense Retrieval: Embeddings + FAISS

- an embedding model (e.g., `sentence-transformers`)
- FAISS (`faiss-cpu`)

In [8]:
import faiss
from sentence_transformers import SentenceTransformer

In [9]:
def build_faiss_index(docs: List[Dict[str, Any]], model_name: str = "all-MiniLM-L6-v2"):
    if faiss is None or SentenceTransformer is None:
        raise RuntimeError("Missing dependencies: faiss and/or sentence-transformers.")

    model = SentenceTransformer(model_name)
    texts = [d["text"] for d in docs]
    emb = model.encode(texts, normalize_embeddings=True, show_progress_bar=True)

    dim = emb.shape[1]
    index = faiss.IndexFlatIP(dim)  # cosine similarity via inner product on normalized vectors
    index.add(emb)

    return model, index, emb

docs = read_jsonl(OUT_DIR / "chunks.jsonl")
model, faiss_index, doc_embeddings = build_faiss_index(docs)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 258.32it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 373/373 [02:31<00:00,  2.46it/s]


In [10]:
def faiss_search(query: str, model, index, docs: List[Dict[str, Any]], k: int = 5):
    q = model.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(q, k)
    results = []
    for score, i in zip(scores[0].tolist(), idxs[0].tolist()):
        if i == -1:
            continue
        results.append((docs[i], float(score)))
    return results

faiss_search("luxury compact coupe 335 hp", model, faiss_index, docs, k=3)

[({'doc_id': 'the_1992_mercedes_benz_300_class_00180',
   'title': 'The 1992 Mercedes-Benz 300-Class',
   'text': 'The 1992 Mercedes-Benz 300-Class is a midsize coupe.\nIt features a 6-cylinder regular unleaded engine producing 217 horsepower.\nThe transmission is automatic with rear drive.\nIt has 2 doors.\nThis model falls under the following categories: luxury.\nFuel efficiency is rated at 15 city MPG and 21 highway MPG.\nIt has a popularity score of 617.\nThe MSRP for this vehicle is $2,248.'},
  0.5663812160491943),
 ({'doc_id': 'the_2004_maserati_coupe_03030',
   'title': 'The 2004 Maserati Coupe',
   'text': 'The 2004 Maserati Coupe is a compact coupe.\nIt features a 8-cylinder premium unleaded engine producing 390 horsepower.\nThe transmission is manual with rear drive.\nIt has 2 doors.\nThis model falls under the following categories: exotic, luxury, high-performance.\nFuel efficiency is rated at 10 city MPG and 15 highway MPG.\nIt has a popularity score of 238.\nThe MSRP for 

## Hybrid


In [11]:
def _minmax_norm(scores: Dict[str, float], eps: float = 1e-9) -> Dict[str, float]:
    """
    Min-max normalize scores into [0, 1] over the provided dict.
    If all scores are equal, returns 0.0 for all.
    """
    if not scores:
        return {}
    vals = list(scores.values())
    lo, hi = min(vals), max(vals)
    if abs(hi - lo) < eps:
        return {k: 0.0 for k in scores}
    return {k: (v - lo) / (hi - lo) for k, v in scores.items()}

def hybrid_search(
    query: str,
    docs: List[Dict[str, Any]],
    bm25_index: BM25Index,
    embed_model,
    faiss_index,
    k: int = 5,
    w_bm25: float = 0.5,
    w_dense: float = 0.5,
    candidate_multiplier: int = 3,
):
    """
    Hybrid retrieval = BM25 + Dense (FAISS) with score fusion.

    - Retrieves top (k * candidate_multiplier) from each retriever
    - Normalizes each retriever's scores via min-max
    - Fuses scores: w_bm25 * bm25_norm + w_dense * dense_norm
    - Returns top-k fused results: List[(doc, fused_score, bm25_score, dense_score)]
    """
    k0 = max(k * candidate_multiplier, k)

    # Get candidates from both retrievers
    bm25_res = bm25_index.search(query, k=k0)                # [(doc, bm25_score), ...]
    dense_res = faiss_search(query, embed_model, faiss_index, docs, k=k0)  # [(doc, dense_score), ...]

    # Map to doc_id -> raw score
    bm25_scores = {d["doc_id"]: s for d, s in bm25_res}
    dense_scores = {d["doc_id"]: s for d, s in dense_res}

    # Normalize scores within each retriever’s candidate set
    bm25_norm = _minmax_norm(bm25_scores)
    dense_norm = _minmax_norm(dense_scores)

    # Union doc_ids from both sets
    all_ids = set(bm25_scores.keys()) | set(dense_scores.keys())

    # Build doc lookup (doc_id -> doc)
    doc_by_id = {d["doc_id"]: d for d in docs}

    # Fuse
    fused = []
    for doc_id in all_ids:
        b_raw = bm25_scores.get(doc_id, 0.0)
        d_raw = dense_scores.get(doc_id, 0.0)
        b = bm25_norm.get(doc_id, 0.0)
        d = dense_norm.get(doc_id, 0.0)
        fused_score = w_bm25 * b + w_dense * d

        doc = doc_by_id.get(doc_id)
        if doc is None:
            continue

        fused.append((doc, float(fused_score), float(b_raw), float(d_raw)))

    fused.sort(key=lambda x: x[1], reverse=True)
    return fused[:k]

In [12]:
results = hybrid_search(
    "rear drive manual 335 horsepower",
    docs=docs,
    bm25_index=bm25,
    embed_model=model,
    faiss_index=faiss_index,
    k=5,
    w_bm25=0.5,
    w_dense=0.5
)

for doc, fused, b_raw, d_raw in results:
    print(doc["title"])
    print(" fused:", round(fused, 4), "| bm25:", round(b_raw, 4), "| dense:", round(d_raw, 4))
    print("-" * 60)

The 1992 Mercedes-Benz 300-Class
 fused: 0.5 | bm25: 0.0 | dense: 0.5424
------------------------------------------------------------
The 2015 BMW Z4
 fused: 0.5 | bm25: 7.3401 | dense: 0.0
------------------------------------------------------------
The 2016 BMW Z4
 fused: 0.5 | bm25: 7.3401 | dense: 0.0
------------------------------------------------------------
The 2014 BMW Z4
 fused: 0.5 | bm25: 7.3401 | dense: 0.0
------------------------------------------------------------
The 1993 Mercedes-Benz 300-Class
 fused: 0.4836 | bm25: 0.0 | dense: 0.5418
------------------------------------------------------------


## Reranker

In [13]:
# Reranker + unified retrieval
from sentence_transformers import CrossEncoder

class CrossEncoderReranker:
    def __init__(self, model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"):
        self.model = CrossEncoder(model_name)

    def rerank(
        self,
        query: str,
        docs_and_scores: List[Tuple[Dict[str, Any], float]],
        top_k: int = 5
    ) -> List[Tuple[Dict[str, Any], float]]:
        """
        docs_and_scores: output from any retriever, as [(doc, retriever_score), ...]
        Returns: [(doc, rerank_score), ...] sorted desc
        """
        pairs = [(query, d["text"]) for (d, _) in docs_and_scores]
        rerank_scores = self.model.predict(pairs)

        reranked = []
        for (doc, _), s in zip(docs_and_scores, rerank_scores):
            reranked.append((doc, float(s)))

        reranked.sort(key=lambda x: x[1], reverse=True)
        return reranked[:top_k]


def retrieve_candidates(
    query: str,
    mode: str,
    *,
    docs: List[Dict[str, Any]],
    bm25_index=None,
    faiss_model=None,
    faiss_index=None,
    k_candidates: int = 20,
    k_final: int = 5,
    reranker: CrossEncoderReranker | None = None,
) -> List[Dict[str, Any]]:
    """
    mode: "bm25" | "faiss" | "hybrid"
    Returns: final top-k docs (after optional rerank)
    """
    if mode == "bm25":
        candidates = bm25_index.search(query, k=k_candidates)   # [(doc, score), ...]
    elif mode == "faiss":
        candidates = faiss_search(query, faiss_model, faiss_index, docs, k=k_candidates)
    elif mode == "hybrid":
        # hybrid_search returns (doc, fused, bm25_raw, dense_raw)
        hybrid = hybrid_search(
            query=query,
            docs=docs,
            bm25_index=bm25_index,
            embed_model=faiss_model,
            faiss_index=faiss_index,
            k=k_candidates,
            w_bm25=0.5,
            w_dense=0.5,
        )
        candidates = [(doc, fused) for (doc, fused, _, _) in hybrid]
    else:
        raise ValueError("mode must be one of: bm25, faiss, hybrid")

    # Rerank
    if reranker is not None:
        reranked = reranker.rerank(query, candidates, top_k=k_final)
        return [d for (d, _) in reranked]

    # No rerank: just take top-k from retriever
    return [d for (d, _) in candidates[:k_final]]

In [14]:
reranker = CrossEncoderReranker()

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 273.83it/s, Materializing param=classifier.weight]                                    
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [15]:
q = "What is the MSRP of the 2011 BMW 1 Series M?"
top_docs = retrieve_candidates(
    q, mode="hybrid",
    docs=docs, bm25_index=bm25, faiss_model=model, faiss_index=faiss_index,
    k_candidates=30, k_final=5,
    reranker=reranker
)
[top_docs[i]["doc_id"] for i in range(min(3, len(top_docs)))]

['the_2011_bmw_1_series_m_00000',
 'the_2011_bmw_1_series_00001',
 'the_2011_bmw_1_series_00003']

In [16]:
for mode in ["bm25", "faiss", "hybrid"]:
    top_docs = retrieve_candidates(
        q, mode=mode,
        docs=docs, bm25_index=bm25, faiss_model=model, faiss_index=faiss_index,
        k_candidates=30, k_final=5,
        reranker=reranker
    )
    print(mode, [d["doc_id"] for d in top_docs[:3]])

bm25 ['the_2011_bmw_1_series_m_00000', 'the_2011_bmw_1_series_00001', 'the_2011_bmw_1_series_00003']
faiss ['the_2011_bmw_1_series_m_00000', 'the_2011_bmw_1_series_00001', 'the_2011_bmw_1_series_00003']
hybrid ['the_2011_bmw_1_series_m_00000', 'the_2011_bmw_1_series_00001', 'the_2011_bmw_1_series_00003']


## Reranker evaluation

In [17]:
import json
from tqdm import tqdm

def load_test_queries(path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f]

def recall_at_k(retrieved_ids, relevant_ids):
    return int(len(set(retrieved_ids) & set(relevant_ids)) > 0)

def evaluate_reranker_effect(
    queries,
    mode,
    docs,
    bm25_index,
    faiss_model,
    faiss_index,
    reranker,
    k_candidates=30,
    k_final=5,
):
    """
    Compares:
      1) Raw retriever (top-k_final directly)
      2) Retriever + reranker
    Returns average Recall@k for both
    """
    raw_hits = []
    rerank_hits = []

    for q in tqdm(queries):
        query = q["query"]
        relevant = q.get("relevant_doc_ids", [])

        # ----- Raw retrieval -----
        raw_docs = retrieve_candidates(
            query,
            mode=mode,
            docs=docs,
            bm25_index=bm25_index,
            faiss_model=faiss_model,
            faiss_index=faiss_index,
            k_candidates=k_final,
            k_final=k_final,
            reranker=None
        )
        raw_ids = [d["doc_id"] for d in raw_docs]
        raw_hits.append(recall_at_k(raw_ids, relevant))

        # ----- With reranker -----
        reranked_docs = retrieve_candidates(
            query,
            mode=mode,
            docs=docs,
            bm25_index=bm25_index,
            faiss_model=faiss_model,
            faiss_index=faiss_index,
            k_candidates=k_candidates,
            k_final=k_final,
            reranker=reranker
        )
        reranked_ids = [d["doc_id"] for d in reranked_docs]
        rerank_hits.append(recall_at_k(reranked_ids, relevant))

    return {
        "raw_recall@k": sum(raw_hits) / len(raw_hits),
        "rerank_recall@k": sum(rerank_hits) / len(rerank_hits)
    }

In [18]:
queries = load_test_queries("../data/test/test_query_set.jsonl")

for mode in ["bm25", "faiss", "hybrid"]:
    result = evaluate_reranker_effect(
        queries,
        mode=mode,
        docs=docs,
        bm25_index=bm25,
        faiss_model=model,
        faiss_index=faiss_index,
        reranker=reranker,
        k_candidates=30,
        k_final=5,
    )
    print(mode, result)

100%|██████████| 40/40 [00:35<00:00,  1.14it/s]


bm25 {'raw_recall@k': 0.425, 'rerank_recall@k': 0.45}


100%|██████████| 40/40 [00:19<00:00,  2.07it/s]


faiss {'raw_recall@k': 0.425, 'rerank_recall@k': 0.45}


100%|██████████| 40/40 [00:35<00:00,  1.13it/s]

hybrid {'raw_recall@k': 0.45, 'rerank_recall@k': 0.45}
